# LangGraph 中的 ToolRuntime

`ToolRuntime` 是 LangGraph 在**执行工具时自动注入**的运行时对象。工具函数只要声明 `runtime: ToolRuntime`，调用方（通常是 `ToolNode`）就会传入它；这个参数不会出现在模型看到的工具参数 schema 中，所以模型不需要、也不能伪造 state、用户身份或工具调用 ID。

它适合让工具读取会话状态、当前请求的运行上下文、长期存储和本次工具调用元数据。本文示例不依赖外部模型/API，直接构造一条工具调用消息，便于观察注入行为。

> `ToolRuntime` 面向工具；图节点常用的是 `langgraph.runtime.Runtime`。两者共享 context、store、stream_writer 等能力，而 ToolRuntime 另外提供 state、config、tool_call_id 和可用工具列表。

## 1. 一个最小可运行例子：读取 state 与 context

- `state`：当前图状态（如消息历史、用户 ID）；它是**本次会话/图运行**的数据。
- `context`：调用方在启动图时传入的运行上下文，适合依赖注入，例如用户权限、租户信息；它通常不是模型生成的参数。
- `tool_call_id`：当前 AI 消息中这次工具调用的 ID。工具结果要与之对应，`ToolNode` 会负责配对。

### `context_schema` 的作用

`StateGraph(..., context_schema=...)` 用来声明这张图接受的**运行时上下文结构及类型**。它让图和工具的依赖更明确，也方便类型检查与 IDE 提示；它本身不负责创建上下文，也不会替你做权限校验。

本例中：

```python
class RequestContext(TypedDict):
    tenant: str
    allowed_cities: list[str]

builder = StateGraph(DemoState, context_schema=RequestContext)
```

`RequestContext` 声明调用时的 context 应包含租户名称和允许查询的城市。运行图时由调用方提供具体值：

```python
weather_graph.invoke(
    state,
    context={"tenant": "华东业务组", "allowed_cities": ["北京", "上海"]},
)
```

工具通过 `runtime.context` 读取这份调用级数据。比如工具可以先判断 `city` 是否在 `allowed_cities` 中，再决定是否查询。**schema 是类型/结构契约，不是安全边界**：权限逻辑仍需在工具或业务层明确检查。

| 数据 | 由谁提供 | 生命周期与常见用途 |
|---|---|---|
| `runtime.state` | 图状态输入及各节点更新 | 会话/图执行中的消息、用户 ID、当前任务状态 |
| `runtime.context` | 每次调用图时的 `context=...` | 请求级依赖、租户配置、权限范围；通常不作为对话状态保存 |
| `runtime.store` | 编译图时提供 `store=...` | 需要跨 thread 保存和检索的长期数据 |

工具签名中的普通参数（这里的 `city`）会显示给模型；`runtime` 是框架注入参数，不显示给模型。这样模型只负责提出业务输入，调用方提供的上下文则由运行时安全地传给工具。

In [1]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, ToolRuntime


# 定义图状态。messages 是 LangGraph 工具调用约定使用的消息列表。
class DemoState(TypedDict):
    messages: list
    user_id: str


# context_schema 描述调用方提供的运行上下文结构。
class RequestContext(TypedDict):
    tenant: str
    allowed_cities: list[str]


@tool
def get_weather(city: str, runtime: ToolRuntime[RequestContext, DemoState]) -> str:
    """查询天气；示例用固定数据模拟天气服务。"""
    # 普通参数 city 来自模型的工具调用参数。
    # runtime.state 来自图当前状态，模型无需重复传入 user_id。
    user_id = runtime.state["user_id"]

    # runtime.context 来自 graph.invoke(..., context=...)，可用于租户配置、权限校验等。
    request_context = runtime.context
    if city not in request_context["allowed_cities"]:
        return f"用户 {user_id} 无权查询 {city}（租户：{request_context['tenant']}）"

    # 这里仅模拟返回值；真实项目可在此调用天气 API。
    weather = {"北京": "晴，25°C", "上海": "多云，22°C"}.get(city, "天气数据暂缺")
    return f"{city}：{weather}（租户：{request_context['tenant']}）"


# 用 StateGraph + ToolNode 执行工具。ToolNode 会解析 AIMessage 中的 tool_calls，
# 自动构造 ToolRuntime，并将工具结果作为 ToolMessage 写回 messages。
builder = StateGraph(DemoState, context_schema=RequestContext)
builder.add_node("tools", ToolNode([get_weather]))
builder.add_edge(START, "tools")
builder.add_edge("tools", END)
weather_graph = builder.compile()

# 为了演示免去模型：手工构造模型通常会产生的 AIMessage 工具调用。
# 重要：tool_call_id 由模型消息携带，工具本身只從 runtime 读取，不作为模型参数。
example_state = {
    "user_id": "user-001",
    "messages": [AIMessage(content="", tool_calls=[{
        "name": "get_weather",
        "args": {"city": "北京"},
        "id": "call-weather-001",
        "type": "tool_call",
    }])],
}

result = weather_graph.invoke(
    example_state,
    context={"tenant": "华东业务组", "allowed_cities": ["北京", "上海"]},
)
result["messages"][-1]  # ToolMessage，含天气文本和匹配的 tool_call_id

ToolMessage(content='北京：晴，25°C（租户：华东业务组）', name='get_weather', tool_call_id='call-weather-001')

## 2. 使用 store 保存跨会话数据

`runtime.store` 是编译图时传入的持久化 Store。它与 state 的生命周期不同：state 通常属于一次会话或 thread；store 可通过 namespace/key 在不同 thread 间共享数据。读写 store 时要清楚地设计 namespace，避免不同用户的数据混在一起。

下面的偏好工具演示 `get` / `put`。`InMemoryStore` 只在当前 Python 进程内存活，适合示例和测试；生产场景应根据部署选用持久化 store。

In [2]:
from langgraph.store.memory import InMemoryStore


@tool
def save_preference(key: str, value: str, runtime: ToolRuntime[RequestContext, DemoState]) -> str:
    """保存当前用户的一项偏好，例如语言或回答风格。"""
    if runtime.store is None:
        # Store 是可选依赖；如果编译图时没提供，明确返回原因。
        return "当前图没有配置 store，无法保存偏好。"

    # namespace 用 user_id 隔离不同用户；同一 namespace 下通过 key 区分条目。
    namespace = ("users", runtime.state["user_id"], "preferences")
    runtime.store.put(namespace, key, value)
    return f"已保存偏好：{key}={value}"


@tool
def read_preference(key: str, runtime: ToolRuntime[RequestContext, DemoState]) -> str:
    """读取当前用户已经保存的偏好。"""
    if runtime.store is None:
        return "当前图没有配置 store。"

    namespace = ("users", runtime.state["user_id"], "preferences")
    item = runtime.store.get(namespace, key)
    return f"{key}={item.value}" if item else f"尚未设置偏好：{key}"


# 两个工具放进同一个 ToolNode。编译时提供 store 后，工具运行时即可访问它。
preference_store = InMemoryStore()
pref_builder = StateGraph(DemoState, context_schema=RequestContext)
pref_builder.add_node("tools", ToolNode([save_preference, read_preference]))
pref_builder.add_edge(START, "tools")
pref_builder.add_edge("tools", END)
pref_graph = pref_builder.compile(store=preference_store)


def invoke_tool(tool_name: str, args: dict, call_id: str):
    """构造一条单工具调用并运行；便于分别检查保存和读取结果。"""
    return pref_graph.invoke(
        {
            "user_id": "user-001",
            "messages": [AIMessage(content="", tool_calls=[{
                "name": tool_name,
                "args": args,
                "id": call_id,
                "type": "tool_call",
            }])],
        },
        context={"tenant": "华东业务组", "allowed_cities": ["北京", "上海"]},
    )

saved = invoke_tool("save_preference", {"key": "language", "value": "中文"}, "call-save-1")
read = invoke_tool("read_preference", {"key": "language"}, "call-read-1")
read["messages"][-1]  # ToolMessage 内容应为“language=中文”

ToolMessage(content='language=中文', name='read_preference', tool_call_id='call-read-1')

## 3. `stream_writer`：从工具发出进度事件

对于耗时工具，可以通过 `runtime.stream_writer(...)` 发出自定义进度数据。调用方用 `stream_mode="custom"` 订阅这些事件。它是进度/状态提示通道，不会自动替代工具最终返回值；工具完成后仍会产生 ToolMessage。

下面的示例人为分成几个步骤，不访问网络。

In [3]:
@tool
def build_report(topic: str, runtime: ToolRuntime[RequestContext, DemoState]) -> str:
    """分步骤生成一份示例报告，并持续发出进度事件。"""
    # stream_writer 写出的对象由上层以 custom stream mode 接收。
    runtime.stream_writer({"stage": "started", "topic": topic, "percent": 0})

    # 真实应用可以在这些阶段执行检索、汇总、格式化等耗时任务。
    runtime.stream_writer({"stage": "research", "topic": topic, "percent": 40})
    runtime.stream_writer({"stage": "draft", "topic": topic, "percent": 80})

    report = f"关于“{topic}”的示例报告：已完成资料整理与摘要。"
    runtime.stream_writer({"stage": "completed", "topic": topic, "percent": 100})
    return report


report_builder = StateGraph(DemoState, context_schema=RequestContext)
report_builder.add_node("tools", ToolNode([build_report]))
report_builder.add_edge(START, "tools")
report_builder.add_edge("tools", END)
report_graph = report_builder.compile()

report_input = {
    "user_id": "user-001",
    "messages": [AIMessage(content="", tool_calls=[{
        "name": "build_report",
        "args": {"topic": "LangGraph ToolRuntime"},
        "id": "call-report-1",
        "type": "tool_call",
    }])],
}

# stream_mode="custom" 接收 runtime.stream_writer 发出的自定义事件。
# 每项的格式是 (模式名, 数据)，这里只打印 custom 事件，忽略其他流事件。
for mode, chunk in report_graph.stream(
    report_input,
    context={"tenant": "华东业务组", "allowed_cities": ["北京", "上海"]},
    stream_mode=["custom"],
):
    if mode == "custom":
        print(chunk)

# 若想同时看图状态更新，可把 stream_mode 改为 ["custom", "updates"]。

{'stage': 'started', 'topic': 'LangGraph ToolRuntime', 'percent': 0}
{'stage': 'research', 'topic': 'LangGraph ToolRuntime', 'percent': 40}
{'stage': 'draft', 'topic': 'LangGraph ToolRuntime', 'percent': 80}
{'stage': 'completed', 'topic': 'LangGraph ToolRuntime', 'percent': 100}


## 4. tool_call_id、返回值与 Command 更新

`runtime.tool_call_id` 标识当前模型发出的工具调用。平常返回字符串即可，`ToolNode` 会自动生成关联的 ToolMessage。若需要在工具执行后修改图状态或控制跳转，可从工具返回 `Command`；工具返回 Command 时通常要在 `update` 的 `messages` 中显式追加带正确 `tool_call_id` 的 ToolMessage，确保消息历史中的工具调用闭合。

另外，runtime 中还有 `config`（当前 Runnable 配置）和 `tools`（当前可用工具列表）。它们主要用于读取调用配置或编写可复用的工具逻辑。不要把 `config` 当成普通工具参数声明；运行时相关数据应通过 `ToolRuntime` 获取。

In [4]:
from typing import Annotated
import operator

from langchain_core.messages import ToolMessage
from langgraph.types import Command


# Command 要更新 facts 字段，因此在状态 schema 中显式声明它。
# operator.add 作为 reducer，让新事实追加到已有列表，而不是覆盖旧列表。
class FactState(TypedDict):
    messages: list
    user_id: str
    facts: Annotated[list[str], operator.add]


@tool
def remember_fact(fact: str, runtime: ToolRuntime[RequestContext, FactState]) -> Command:
    """把一条事实追加到图状态，并返回与当前调用配对的工具消息。"""
    # 这里返回 Command 来同时更新 state 和 messages。
    # tool_call_id 来自 runtime，确保 ToolMessage 对应发起它的 AI tool call。
    return Command(
        update={
            "facts": [fact],
            "messages": [ToolMessage(
                content=f"已记录：{fact}",
                tool_call_id=runtime.tool_call_id,
                name="remember_fact",
            )],
        }
    )


fact_builder = StateGraph(FactState, context_schema=RequestContext)
fact_builder.add_node("tools", ToolNode([remember_fact]))
fact_builder.add_edge(START, "tools")
fact_builder.add_edge("tools", END)
fact_graph = fact_builder.compile()

fact_result = fact_graph.invoke(
    {
        "user_id": "user-001",
        "facts": [],
        "messages": [AIMessage(content="", tool_calls=[{
            "name": "remember_fact",
            "args": {"fact": "用户偏好使用中文"},
            "id": "call-fact-1",
            "type": "tool_call",
        }])],
    },
    context={"tenant": "华东业务组", "allowed_cities": ["北京", "上海"]},
)
print(fact_result["facts"])              # ['用户偏好使用中文']
print(fact_result["messages"][-1])       # ToolMessage，tool_call_id 匹配

# runtime.tool_call_id 也可用于日志追踪、幂等键或审计记录。
# runtime.config 可读取当前 RunnableConfig；runtime.tools 可枚举本次可用工具。
# 例如：print(runtime.tool_call_id), print(runtime.config), print([t.name for t in runtime.tools])


['用户偏好使用中文']
content='已记录：用户偏好使用中文' name='remember_fact' tool_call_id='call-fact-1'


## 5. `wrap_tool_call`：在工具执行前后加一层拦截

`ToolRuntime` 是**工具内部**读运行时数据的方式；`wrap_tool_call` 是 **Agent middleware 提供的工具调用包装钩子**。它让你在工具实际执行前后统一介入，例如记录日志、计时、重试、改写调用参数，或把异常转换成模型可继续处理的 `ToolMessage`。

处理函数接收 `request` 和 `handler`：

- `request.tool_call`：模型发起的工具调用（名称、参数、调用 ID）。
- `request.tool`：实际要执行的工具对象。
- `request.state`：当前 agent state。
- `request.runtime`：本次调用的运行时对象。
- `handler(request)`：继续执行当前工具调用；包装器可以在调用它前后处理。若不调用 handler，就由包装器自行返回 `ToolMessage` 或 `Command`。

下面的包装器将工具异常转成 `ToolMessage`，并使用原调用的 ID，使 agent 能收到工具错误并决定下一步。

In [7]:
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage


# 在这个完整示例单元中再次定义 middleware，确保单独运行本单元也不会 NameError。
@wrap_tool_call
def handle_tool_errors(request, handler):
    """捕获工具异常，并将异常转为与原工具调用配对的 ToolMessage。"""
    tool_name = request.tool_call["name"]
    tool_call_id = request.tool_call["id"]
    try:
        return handler(request)  # 继续执行真实工具；正常时直接返回工具结果。
    except Exception as exc:
        return ToolMessage(
            content=f"工具 {tool_name} 执行失败：{type(exc).__name__}。请尝试其他方式。",
            tool_call_id=tool_call_id,
            name=tool_name,
        )


from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openrouter import ChatOpenRouter
load_dotenv(override=True)

@tool
def fragile_lookup(item: str) -> str:
    """查询示例服务中的条目；本例故意抛异常以演示包装器。"""
    # 模拟网络超时、第三方服务不可用等工具执行错误。
    raise TimeoutError(f"查询 {item} 时服务超时")


# 使用支持 tool calling 的模型。请先配置 OPENROUTER_API_KEY。
# 若项目中已经有可用的模型实例，也可以直接复用它替换 ChatOpenRouter。
model = ChatOpenRouter(model="inclusionai/ling-3.0-flash-sante:free")

# 关键：middleware=[handle_tool_errors] 将前面定义的包装器挂到 agent 的工具调用流程。
# 没有注册 middleware 时，fragile_lookup 抛出的 TimeoutError 会向上冒泡；
# 注册后，它被转成 ToolMessage，agent 可以把错误当作工具结果继续处理。
agent = create_agent(
    model=model,
    tools=[fragile_lookup],
    middleware=[handle_tool_errors],
)

# 运行后观察 agent 的消息序列：其中会有调用 fragile_lookup 的 AIMessage、
# handle_tool_errors 返回的 ToolMessage，以及模型根据错误生成的后续 AIMessage。
result = agent.invoke({"messages": [{
    "role": "user",
    "content": "请查询 item-123 的信息；如果查询失败，告诉我服务暂时不可用。",
}]})
for message in result["messages"]:
    print(type(message).__name__, message.content)

# 典型流程：模型发起工具调用 -> handler(request) 执行工具并抛 TimeoutError
# -> except 分支返回 ToolMessage -> agent 把 ToolMessage 交回模型 -> 模型给出答复。


HumanMessage 请查询 item-123 的信息；如果查询失败，告诉我服务暂时不可用。
AIMessage 我来为您查询 item-123 的信息。

ToolMessage 工具 fragile_lookup 执行失败：TimeoutError。请尝试其他方式。
AIMessage 很抱歉，查询失败了。服务暂时不可用。


> **Command 示例补充说明**：上面的 `remember_fact` 只展示了 Command 与 tool_call_id 的关联模式。若要把 `fact` 真正累积到状态，应在状态类型中显式声明 `facts` 字段，并定义合适的 reducer（例如 `Annotated[list[str], operator.add]`）；不要向未声明该字段的状态随意写入。

## 6. 接入真实模型时的结构

在真实 agent 中，模型负责决定要不要调用工具，并产生 `AIMessage.tool_calls`；`ToolNode` 根据名称执行对应工具，`ToolRuntime` 则由 LangGraph 在执行时注入。典型循环是：模型节点 → `tools_condition` 判断是否有工具调用 → ToolNode → 回到模型节点。工具参数 schema 只包含业务输入，例如 `city`；state/context/store 等运行时依赖由框架注入。

```python
# 常见结构示意（model 需是支持 tool calling 的 chat model）：
# tools = [get_weather]
# model_with_tools = model.bind_tools(tools)
# builder.add_node("llm", lambda state: {"messages": [model_with_tools.invoke(state["messages"])]})
# builder.add_node("tools", ToolNode(tools))
# builder.add_conditional_edges("llm", tools_condition)
# builder.add_edge("tools", "llm")
# graph = builder.compile(store=store)
```

### 使用时记住

1. **声明类型触发注入**：工具参数写 `runtime: ToolRuntime[...]`，不要把 runtime 字段放进模型的调用参数。
2. **短期与长期分开**：会话内数据从 `runtime.state` 读；跨 thread 数据通常通过 `runtime.store` 读写。
3. **外部依赖通过 context 注入**：调用方传入 `context=...`，工具从 `runtime.context` 读取，便于测试和隔离租户配置。
4. **进度和结果是两条通道**：`stream_writer` 发自定义事件；return 值形成最终工具结果。
5. **Store 需要配置**：编译图时传入 `store=...`，否则 `runtime.store` 可能为 `None`。
6. **访问要校验**：state/context/store 可能包含用户数据或权限信息，工具应做必要的权限和输入校验。